In [ ]:
import csv
import math
import nltk
nltk.download('semcor')
from nltk.corpus import wordnet as wn
import numpy as np
import re
from nltk.corpus import semcor
import random

[nltk_data] Downloading package semcor to /root/nltk_data...


In [ ]:
def remove_non_word_chars(sentence):
    return re.sub(r'\W+', ' ', sentence).strip()

def ComputeOverlap(signature,context):
    print("OVERLAP")
    print("SIGNATURE",signature)
    print("CONTEXT",context)
    inter = signature.intersection(context)
    print("INTERSECTION",inter)
    return len(inter)

#read file stopwords
stop_words = []
with open('stop_words_FULL.txt') as f:
    stop_words = f.read().splitlines()

""" stop_words = set([
        "a", "about", "above", "after", "again", "against", "all", "am", "an",
        "and", "any", "are", "as", "at", "be", "because", "been", "before",
        "being", "below", "between", "both", "but", "by", "could", "did",
        "do", "does", "doing", "down", "during", "each", "few", "for", "from",
        "further", "had", "has", "have", "having", "he", "her", "here",
        "hers", "herself", "him", "himself", "his", "how", "i", "if", "in",
        "into", "is", "it", "its", "itself", "just", "me", "more", "most",
        "my", "myself", "no", "nor", "not", "now", "of", "on", "once", "only",
        "or", "other", "our", "ours", "ourselves", "out", "over", "own",
        "same", "she", "should", "so", "some", "such", "than", "that", "the",
        "their", "theirs", "them", "themselves", "then", "there", "these",
        "they", "this", "those", "through", "to", "too", "under", "until",
        "up", "very", "was", "we", "were", "what", "when", "where", "which",
        "while", "who", "whom", "why", "will", "with", "you", "your", "yours",
        "yourself", "yourselves"
    ]) """

def SimplifiedLesk(word,sentence):
    best_sense = None
    max_overlap = 0
    context = set(remove_non_word_chars(sentence).split(" ")).difference(stop_words)
    """ print("CONTEXT",context) """
    for sense in wn.synsets(word):
        if (best_sense is None):
            best_sense = sense
        print("SENSE", sense)
        signature = set()
        for example in sense.examples():
            signature.update(remove_non_word_chars(example).split(" "))
        
        signature.update(remove_non_word_chars(sense.definition()).split(" "))

        signature = signature.difference(stop_words)

        overlap = ComputeOverlap(signature,context)
        if overlap > max_overlap:
            max_overlap = overlap
            best_sense = sense
    return best_sense 
        


best = SimplifiedLesk("bank", "the bank can guarantee deposits will eventually cover future tuition costs because it invests in adjustable-rate mortgage securities")

print(best, best.definition())

SENSE Synset('bank.n.01')
OVERLAP
SIGNATURE {'slope', 'currents', 'sat', 'watched', 'pulled', 'sloping', 'land', 'canoe', 'body', 'river', 'water', 'bank'}
CONTEXT {'adjustable', 'invests', 'guarantee', 'will', 'cover', 'mortgage', 'rate', 'eventually', 'tuition', 'deposits', 'future', 'costs', 'bank', 'securities'}
INTERSECTION {'bank'}
SENSE Synset('depository_financial_institution.n.01')
OVERLAP
SIGNATURE {'lending', 'money', 'holds', 'channels', 'mortgage', 'activities', 'cashed', 'check', 'deposits', 'financial', 'institution', 'accepts', 'bank'}
CONTEXT {'adjustable', 'invests', 'guarantee', 'will', 'cover', 'mortgage', 'rate', 'eventually', 'tuition', 'deposits', 'future', 'costs', 'bank', 'securities'}
INTERSECTION {'bank', 'mortgage', 'deposits'}
SENSE Synset('bank.n.03')
OVERLAP
SIGNATURE {'long', 'earth', 'ridge', 'pile', 'bank', 'huge'}
CONTEXT {'adjustable', 'invests', 'guarantee', 'will', 'cover', 'mortgage', 'rate', 'eventually', 'tuition', 'deposits', 'future', 'costs',

In [23]:
def get_sentence_from_semcor(sentence_num):
   sentence = " ".join(semcor.sents()[sentence_num])
   tags = semcor.tagged_sents(tag="sem")[sentence_num]
   # print(tags)
   word = None
   while(word == None):
        i = random.randint(0, len(tags)-1)
        if tags[i][0] not in stop_words and isinstance(tags[i], nltk.Tree) and isinstance(tags[i][0], str) and isinstance(tags[i].label(), nltk.corpus.reader.wordnet.Lemma):
            word = tags[i][0]
            target = tags[i].label().synset()
   return sentence, word, target

def get_random_semcor_sentences(n=1):
    max_sentence = len(semcor.sents())-1
    list_of_random_indexes = []
    while len(list_of_random_indexes)<n:
        test_index = random.randint(0, max_sentence)
        if test_index not in list_of_random_indexes:
            list_of_random_indexes.append(test_index)

    corpus_sentences = [get_sentence_from_semcor(i) for i in list_of_random_indexes]

    return corpus_sentences


# Esempio di utilizzo della funzione
random_sentences = get_random_semcor_sentences()
disambiguated_sentences = [SimplifiedLesk(sentence[1], sentence[0]) for sentence in random_sentences]
target_sentences = [sentence[2] for sentence in random_sentences]


for sentence, disambiguated, target in zip(random_sentences, disambiguated_sentences, target_sentences):
    print(sentence)
    print(disambiguated)
    print(target) 


SENSE Synset('subtract.v.01')
OVERLAP
SIGNATURE {'paycheck', 'subtraction', 'amount', 'subtract'}
CONTEXT {'number', 'The', 'left', 'subtracted', 'symbol'}
INTERSECTION set()
SENSE Synset('subtract.v.02')
OVERLAP
SIGNATURE {'borrowed', 'French', 'prefix', 'subtracted', 'word'}
CONTEXT {'number', 'The', 'left', 'subtracted', 'symbol'}
INTERSECTION {'subtracted'}
('The number on the right of the symbol is always subtracted from the number on the left of the symbol .', 'subtracted', Synset('subtract.v.01'))
Synset('subtract.v.02')
Synset('subtract.v.01')
